In [ ]:
#| default_exp read

## Reading and inspection

Readable notebook views plus symbol-level context for co-creation.

`nb_overview`, `nb_chapter`, and `nb_cell` split notebook reading into focused tools instead of one broad selector. `nb_overview` maps section headings and definitions, `nb_chapter` shows the notebook head plus one chapter, and `nb_cell` shows the full editing context around one cell.

The focused tools keep cell ids and hashes visible so a later guarded edit can use the same source context.

Reading is the first problem to solve for notebook automation. An agent should not have to inspect raw `.ipynb` JSON just to learn what a notebook contains, and a human reviewer should not have to scroll through outputs and metadata to find the code.

This notebook builds compact views for three common questions: what cells exist, what full source is in a selected cell, and what documentation surrounds a symbol.

The reader is intentionally a triage tool before it is a renderer. Start with `nb_overview(nb_path)`, move to `nb_chapter(nb_path, name=...)` or `nb_chapter(nb_path, any_cell_id=...)` when you need surrounding structure, and use `nb_cell(nb_path, id=...)` when you need line-numbered source for an edit.

```python
nb_overview("nbs/02_write.ipynb")
nb_chapter("nbs/02_write.ipynb", name="Writing and changing")
nb_cell("nbs/02_write.ipynb", query='contains="def update_cell"')
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb, read_nb as _read_nb, write_nb as _write_nb
from nbskill.read import nb_cell as _example_nb_cell
from nbskill.read import nb_chapter as _example_nb_chapter
from nbskill.read import nb_overview as _example_nb_overview
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
path = demo_path("01_read_example.ipynb")
try:
    _write_nb(_new_nb([
        _mk_cell("preamble = True", cell_type="code"),
        _mk_cell("## Demo\nWhy this function exists.", cell_type="markdown"),
        _mk_cell("#| export\ndef answer():\n    \"\"\"Return the demo answer.\"\"\"\n    return 42", cell_type="code"),
        _mk_cell("assert answer() == 42", cell_type="code"),
        _mk_cell("answer()", cell_type="code"),
    ]), path)
    print("nb_overview")
    _example_nb_overview(str(path), include_docs=True)
    print("\nnb_chapter")
    _example_nb_chapter(str(path), name="Demo")
    print("\nnb_cell")
    target = _read_nb(path).cells[2].id
    _example_nb_cell(str(path), id=target)
finally:
    remove_demo_path(path)

In [ ]:
#| export
import ast
import copy
import json
import re
import shlex

from fastcore.nbio import read_nb as _read_nb
from fastcore.script import call_parse

from nbskill.foundation import (
    cell_hash, cell_matches_type, cell_prefix, cell_source, chapter_index_set,
    cli_return, find_cell_by_id, first_line, is_definition_node,
    is_export_directive, is_exported_code_cell, matches_filter, tracked_call,
    with_context,
)

### Output shapes

The focused readers serve three attention levels. `nb_overview` is an unnumbered map of headings and definitions, `nb_chapter` is an unnumbered chapter view with the notebook head included, and `nb_cell` is the only reader that prints line numbers because it is the edit-oriented view.

In [ ]:
#| export
def _format_overview(items, show_ids=False):
    lines = []
    for idx, cell in items:
        summary = first_line(cell.source)
        lines.append(f"{cell_prefix(idx, cell, show_ids)} | {summary}")
    return "\n".join(lines)

In [ ]:
#| export
def _markdown_overview(cell, include_docs=False):
    source = cell_source(cell).strip()
    headings = []
    for line in source.splitlines():
        text = line.strip()
        if re.match(r"^#{1,6}\s+", text): headings.append(text)
    if headings: return headings
    return source.splitlines() if include_docs and source else []


def _definition_lines(node, indent=""):
    tmp = copy.deepcopy(node)
    tmp.body = [ast.Pass()]
    ast.fix_missing_locations(tmp)
    lines = []
    for line in ast.unparse(tmp).splitlines():
        if line.strip() == "pass": continue
        lines.append(f"{indent}{line}" if line else line)
    return lines


def _docstring_lines(node, indent="    "):
    doc = ast.get_docstring(node)
    if not doc: return []
    quote = indent + chr(34) * 3
    return [quote, *[f"{indent}{line}" for line in doc.splitlines()], quote]


def _function_overview(node, indent=""):
    return [*_definition_lines(node, indent=indent), *_docstring_lines(node, indent=indent + "    ")]


def _class_overview(node):
    lines = [*_definition_lines(node), *_docstring_lines(node)]
    methods = [child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef))]
    for method in methods:
        if lines and lines[-1] != "": lines.append("")
        lines += _function_overview(method, indent="    ")
    return lines


def _code_overview(cell):
    try: tree = ast.parse(cell.source)
    except SyntaxError: return []
    lines = []
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)): lines.append(ast.unparse(node))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): lines += _function_overview(node)
        elif isinstance(node, ast.ClassDef): lines += _class_overview(node)
        if lines and lines[-1] != "": lines.append("")
    if lines and lines[-1] == "": lines.pop()
    return lines


def _format_headers(items, include_docs=False):
    chunks = []
    for idx, cell in items:
        if cell.cell_type == "markdown": lines = _markdown_overview(cell, include_docs=include_docs)
        elif cell.cell_type == "code": lines = _code_overview(cell)
        else: lines = []
        if lines: chunks.append(f"{cell_prefix(idx, cell, True)}\n" + "\n".join(lines))
    return "\n\n".join(chunks)

In [ ]:
#| export
def _format_source(source, line_numbers=False):
    if not line_numbers: return source
    lines = source.splitlines() or [""]
    return "\n".join(f"{idx} | {line}" for idx, line in enumerate(lines, start=1))


def _format_full(items, show_ids=False, line_numbers=False):
    chunks = []
    for idx, cell in items:
        chunks.append(f"{cell_prefix(idx, cell, show_ids)}\n{_format_source(cell_source(cell), line_numbers=line_numbers)}")
    return "\n\n".join(chunks)

### A small query language

Automation needs stable selectors, but humans need short commands. The query helpers accept aliases like `id`, `type`, `chapter`, and `contains`, then normalize them into one internal selection shape.

In [ ]:
#| export
_QUERY_KEY_ALIASES = {
    "id": "cell_id",
    "cell": "cell_id",
    "cell_id": "cell_id",
    "chapter": "chapter",
    "type": "cell_type",
    "class": "cell_type",
    "cell_type": "cell_type",
    "contains": "contains",
    "text": "contains",
    "regex": "regex",
    "re": "regex",
}


def _normalize_query_key(key):
    name = _QUERY_KEY_ALIASES.get(str(key).strip().lower())
    if name is None:
        choices = ", ".join(sorted(_QUERY_KEY_ALIASES))
        raise ValueError(f"Unknown query key {key!r}; use one of: {choices}")
    return name


def _normalize_query_dict(spec):
    return {_normalize_query_key(key): None if value is None else str(value) for key, value in dict(spec).items()}


def _parse_query_terms(text):
    spec, bare = {}, []
    for term in shlex.split(str(text)):
        sep = "=" if "=" in term else ":" if ":" in term else None
        if sep is None:
            bare.append(term)
            continue
        key, value = term.split(sep, 1)
        spec[_normalize_query_key(key)] = value
    if bare and "contains" not in spec: spec["contains"] = " ".join(bare)
    return spec


def _parse_query(query):
    if query is None: return [{}]
    if isinstance(query, dict): return [_normalize_query_dict(query)]
    if isinstance(query, (list, tuple)):
        specs = []
        for item in query: specs.extend(_parse_query(item))
        return specs or [{}]

    text = str(query).strip()
    if not text: return [{}]
    try: parsed = json.loads(text)
    except json.JSONDecodeError:
        return [_parse_query_terms(part) for part in text.split(";") if part.strip()]
    return _parse_query(parsed)



def _select_query_items(nb, spec):
    items = [find_cell_by_id(nb.cells, spec["cell_id"])] if spec.get("cell_id") else list(enumerate(nb.cells))
    if spec.get("chapter") is not None:
        chapter_idxs = chapter_index_set(nb.cells, spec["chapter"])
        items = [(i, c) for i, c in items if i in chapter_idxs]
    if spec.get("cell_type") is not None: items = [(i, c) for i, c in items if cell_matches_type(c, spec["cell_type"])]
    if spec.get("contains") is not None: items = [(i, c) for i, c in items if spec["contains"] in c.source]
    if spec.get("regex") is not None: items = [(i, c) for i, c in items if matches_filter(c.source, spec["regex"])]
    return items



def _cell_defined_symbols(cell):
    if getattr(cell, "cell_type", None) != "code": return []
    try: tree = ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): symbols.append(node.name)
        if isinstance(node, ast.ClassDef):
            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)): symbols.append(f"{node.name}.{child.name}")
    return symbols


def _format_usage_for_items(path, items):
    symbols = []
    for _, cell in items: symbols.extend(_cell_defined_symbols(cell))
    if not symbols: return ""
    try:
        from nbskill.graph import symbol_usage_summary
        return symbol_usage_summary(path, symbols)
    except Exception as exc:
        return f"Usage unavailable: {type(exc).__name__}: {exc}"

### The public notebook readers

The public surface is intentionally small. `nb_overview` accepts a notebook path, always returns Markdown headings plus imports, function signatures, class signatures, method signatures, and their docstrings, and can include ordinary non-heading Markdown cells with `include_docs=True`. Its `verbose` flag controls printing for CLI/Python usage and is intentionally not exposed through the MCP tool. `nb_chapter` selects a chapter by query, title, or any cell id inside it. `nb_cell` selects exactly one cell by query or id and returns the line-numbered editing context.

In [ ]:
#| export
def _chapter_title_from_cell(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in cell_source(cell).splitlines():
        match = re.match(r"^##\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None


def _chapter_spans_for_nb(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title_from_cell(cell))]
    if not starts: return [dict(title="Notebook", start=0, end=len(cells))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans


def _notebook_head_items(cells):
    spans = _chapter_spans_for_nb(cells)
    head_end = spans[0]["start"] if spans else len(cells)
    return [(idx, cells[idx]) for idx in range(head_end)]


def _chapter_span_for_index(cells, idx):
    spans = _chapter_spans_for_nb(cells)
    if spans and idx < spans[0]["start"]:
        return dict(title="Notebook head", start=0, end=spans[0]["start"])
    for span in spans:
        if span["start"] <= idx < span["end"]: return span
    raise ValueError(f"Cell index {idx} is outside the notebook")


def _one_chapter_span(cells, name):
    matches = [span for span in _chapter_spans_for_nb(cells) if matches_filter(span["title"], name)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No chapter matches {name!r}")
    titles = ", ".join(span["title"] for span in matches)
    raise ValueError(f"Chapter {name!r} matches multiple chapters: {titles}")


def _query_items(nb, query):
    items, seen = [], set()
    for spec in _parse_query(query):
        for idx, cell in _select_query_items(nb, spec):
            if idx in seen: continue
            seen.add(idx)
            items.append((idx, cell))
    return items


def _selected_chapter_span(nb, query=None, name=None, any_cell_id=None):
    selectors = [value is not None for value in (query, name, any_cell_id)]
    if sum(selectors) != 1: raise ValueError("Pass exactly one of query, name, or any_cell_id")
    if name is not None: return _one_chapter_span(nb.cells, name)
    if any_cell_id is not None:
        idx, _ = find_cell_by_id(nb.cells, any_cell_id)
        return _chapter_span_for_index(nb.cells, idx)
    items = _query_items(nb, query)
    if not items: raise ValueError(f"No cells match query {query!r}")
    return _chapter_span_for_index(nb.cells, items[0][0])


def _chapter_items(nb, span):
    idxs = set()
    items = []
    for idx, cell in [*_notebook_head_items(nb.cells), *[(i, nb.cells[i]) for i in range(span["start"], span["end"])]]:
        if idx in idxs: continue
        idxs.add(idx)
        items.append((idx, cell))
    return items


def _selected_cell_item(nb, query=None, id=None):
    if (query is None) == (id is None): raise ValueError("Pass exactly one of query or id")
    if id is not None: return find_cell_by_id(nb.cells, id)
    items = _query_items(nb, query)
    if len(items) == 1: return items[0]
    if not items: raise ValueError(f"No cells match query {query!r}")
    preview = _format_overview(items[:12], show_ids=True)
    raise ValueError(f"Query {query!r} matched {len(items)} cells; narrow it or pass id.\n{preview}")


@call_parse
@tracked_call
def nb_overview(
    nb_path: str,  # Notebook path
    include_docs: bool = False,  # Include Markdown cells without headings
    verbose: bool = True,  # Print the overview; pass False to only return it
):
    "Print a focused notebook map of headings, imports, signatures, and docstrings."
    nb = _read_nb(nb_path)
    text = _format_headers(list(enumerate(nb.cells)), include_docs=include_docs)
    if verbose and text: print(text)
    return cli_return(text)


@call_parse
@tracked_call
def nb_chapter(
    nb_path: str,  # Notebook path
    query: str | None = None,  # Query for any cell inside the chapter
    name: str | None = None,  # Chapter title string or regex
    any_cell_id: str | None = None,  # Any cell id inside the chapter
):
    "Print the notebook head plus one selected chapter."
    nb = _read_nb(nb_path)
    span = _selected_chapter_span(nb, query=query, name=name, any_cell_id=any_cell_id)
    text = _format_full(_chapter_items(nb, span), show_ids=True, line_numbers=False)
    if text: print(text)
    return cli_return(text)


@call_parse
@tracked_call
def nb_cell(
    nb_path: str,  # Notebook path
    query: str | None = None,  # Query that resolves to exactly one cell
    id: str | None = None,  # Stable notebook cell id
):
    "Print one cell with editing context and caller/callee usage."
    nb = _read_nb(nb_path)
    idx, cell = _selected_cell_item(nb, query=query, id=id)
    items = with_context(nb.cells, [(idx, cell)], include=True)
    text = _format_full(items, show_ids=True, line_numbers=True)
    if usage := _format_usage_for_items(nb_path, [(idx, cell)]):
        text = f"{text}\n\nUsage:\n{usage}" if text else f"Usage:\n{usage}"
    if text: print(text)
    return cli_return(text)

In [ ]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not line.lstrip().startswith("#|"))

In [ ]:
#| export
def _annotation_name(annotation):
    if annotation is None: return None
    if isinstance(annotation, ast.Name): return annotation.id
    if isinstance(annotation, ast.Attribute): return annotation.attr
    if isinstance(annotation, ast.Constant): return annotation.value
    return ast.unparse(annotation)

In [ ]:
#| export
def _first_arg_annotation(node):
    args = node.args.posonlyargs or node.args.args
    return _annotation_name(args[0].annotation) if args else None

In [ ]:
#| export
def _node_defines_symbol(node, symbol):
    parts = symbol.split(".")
    name = parts[-1]
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
        return True
    if len(parts) < 2: return False

    cls_name, meth_name = parts[-2], parts[-1]
    if isinstance(node, ast.ClassDef) and node.name == cls_name:
        return any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == meth_name for child in node.body)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == meth_name:
        return _first_arg_annotation(node) == cls_name
    return False

In [ ]:
#| export
def _cell_defines_symbol(cell, symbol):
    if cell.cell_type != "code": return False
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return False
    return any(_node_defines_symbol(node, symbol) for node in tree.body)

In [ ]:
#| export
def _find_symbol_cell(nb, symbol):
    for idx, cell in enumerate(nb.cells):
        if _cell_defines_symbol(cell, symbol): return idx
    raise ValueError(f"Could not find symbol {symbol!r}")

In [ ]:
#| export
def _find_symbol_node(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return None
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return None
    parts = symbol.split(".")
    for node in tree.body:
        if len(parts) == 1 and isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == symbol:
            return node
        if _node_defines_symbol(node, symbol):
            if isinstance(node, ast.ClassDef) and len(parts) > 1:
                name = parts[-1]
                return next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == name), node)
            return node
    return None

In [ ]:
#| export
def _previous_markdown(cells, idx, limit):
    docs = []
    pos = idx - 1
    while pos >= 0 and len(docs) < limit:
        cell = cells[pos]
        if getattr(cell, "cell_type", None) != "markdown": break
        docs.append((pos, cell))
        pos -= 1
    return list(reversed(docs))

In [ ]:
#| export
def _following_examples(cells, idx, limit):
    examples = []
    pos = idx + 1
    while pos < len(cells) and len(examples) < limit:
        cell = cells[pos]
        if is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) in {"markdown", "code"}: examples.append((pos, cell))
        pos += 1
    return examples

In [ ]:
#| export
def _symbol_signature_text(cell, symbol):
    node = _find_symbol_node(cell, symbol)
    if node is None: return _code_overview(cell)
    if isinstance(node, ast.ClassDef): return _class_overview(node)
    return _function_overview(node)

In [ ]:
#| export
def _usage_group_locations(raw_locations):
    raw = str(raw_locations or "").strip()
    if not raw or raw == "none": return [], 0
    grouped = {}
    for item in [part.strip() for part in raw.split(";") if part.strip()]:
        path, _, cell_id = item.rpartition(" id=")
        if not path: path, cell_id = item, ""
        grouped.setdefault(path, []).append(cell_id)
    return list(grouped.items()), sum(len(ids) for ids in grouped.values())


def _format_usage_locations(label, raw_locations, max_paths=4, max_ids=4):
    groups, total = _usage_group_locations(raw_locations)
    if not total: return [f"{label}: none"]
    cell_word = "cell" if total == 1 else "cells"
    notebook_word = "notebook" if len(groups) == 1 else "notebooks"
    lines = [f"{label}: {total} {cell_word} across {len(groups)} {notebook_word}"]
    for path, ids in groups[:max_paths]:
        shown_ids = [item for item in ids[:max_ids] if item]
        suffix = f": {', '.join(shown_ids)}" if shown_ids else ""
        if len(ids) > max_ids: suffix += f", +{len(ids) - max_ids} more"
        lines.append(f"- {path}{suffix}")
    if len(groups) > max_paths: lines.append(f"- +{len(groups) - max_paths} more notebooks")
    return lines


def _raw_caller_usage_lines(raw_lines):
    if "Caller usages:" not in raw_lines: return []
    start = raw_lines.index("Caller usages:") + 1
    return [line for line in raw_lines[start:] if line.startswith("- ")]


def _format_symbol_usage(path, symbol):
    try:
        from nbskill.graph import symbol_usage_summary
        raw = symbol_usage_summary(path, [symbol])
    except Exception as exc:
        return [f"Usage unavailable: {type(exc).__name__}: {exc}"]
    if not raw: return []
    raw_lines = raw.splitlines()
    line = raw_lines[0]
    prefix = f"{symbol}: callers="
    if not line.startswith(prefix) or "; callees=" not in line: return raw_lines
    callers, _, callees = line[len(prefix):].partition("; callees=")
    lines = _format_usage_locations("Callers", callers)
    caller_usage = _raw_caller_usage_lines(raw_lines)
    if caller_usage:
        lines.append("Caller usages:")
        lines.extend(caller_usage)
    callee_items = [item.strip() for item in callees.split(";") if item.strip() and item.strip() != "none"]
    lines.append(f"Callees: {', '.join(callee_items)}" if callee_items else "Callees: none")
    return lines


def _format_symbol_doc(path, nb, symbol, context=2, source=False, show_ids=False):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    lines = [f"Symbol {symbol}", f"Location: {path} {cell_prefix(idx, cell, show_ids)}"]
    docs = _previous_markdown(nb.cells, idx, context)
    if docs:
        lines.append("")
        lines.append("Docs")
        for doc_idx, doc_cell in docs:
            if show_ids: lines.append(cell_prefix(doc_idx, doc_cell, show_ids))
            lines.append(doc_cell.source.strip())
    signature = _symbol_signature_text(cell, symbol)
    if signature:
        lines.append("")
        lines.append("Definition")
        lines.extend(signature)
    if source:
        lines.append("")
        lines.append("Source cell")
        lines.append(cell.source.strip())
    examples = _following_examples(nb.cells, idx, context)
    if examples:
        lines.append("")
        lines.append("Examples/tests")
        for ex_idx, ex_cell in examples:
            lines.append(cell_prefix(ex_idx, ex_cell, show_ids))
            lines.append(ex_cell.source.strip())
    usage = _format_symbol_usage(path, symbol)
    if usage:
        lines.append("")
        lines.append("Usage")
        lines.extend(usage)
    return "\n".join(lines)

In [ ]:
#| export
@call_parse
@tracked_call
def show_doc(
    path: str,  # Notebook path
    symbol: str | None = None,  # Function, class, or Class.method to inspect
    context: int = 2,  # Nearby doc/example cells to include around the symbol
    source: bool = False,  # Include the full source cell
    show_ids: bool = False,  # Include source hashes in output
):
    "Show rationale/docs, exported code, and show-off examples for a notebook symbol."
    if symbol is None:
        if not isinstance(path, str):
            from nbdev.showdoc import show_doc as _nbdev_show_doc
            return _nbdev_show_doc(path)
        raise ValueError("Pass symbol when path is a notebook")
    nb = _read_nb(path)
    text = _format_symbol_doc(path, nb, symbol, context=context, source=source, show_ids=show_ids)
    print(text)
    return cli_return(text)

In [ ]:
path = demo_path("01_read_doc.ipynb")
try:
    nb = new_nb([
        mk_cell("## Addition\nThis explains the exported symbol.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        show_doc(str(path), "add")
    text = out.getvalue()
    assert "Add two values." in text
    assert "assert add(1, 2) == 3" in text
finally:
    remove_demo_path(path)

In [ ]:
path = demo_path("01_read_sample.ipynb")
try:
    example = mk_cell("add(2, 3)", cell_type="code")
    example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "5"},
    }]
    nb = new_nb([
        mk_cell("Notebook-level note.", cell_type="markdown"),
        mk_cell("import math", cell_type="code"),
        mk_cell("## Math\nTiny rationale.", cell_type="markdown"),
        mk_cell(
            "#| export\ndef add(a, b):\n"
            "    \"\"\"Add values.\"\"\"\n"
            "    return a + b\n\n"
            "class Calculator:\n"
            "    \"\"\"Calculate values.\"\"\"\n"
            "    def __init__(self, base):\n"
            "        \"\"\"Store the base value.\"\"\"\n"
            "        self.base = base\n\n"
            "    def total(self, value):\n"
            "        \"\"\"Add value to the base.\"\"\"\n"
            "        return self.base + value",
            cell_type="code",
        ),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
        example,
    ])
    _write_nb(nb, path)

    out = _StringIO()
    with _redirect_stdout(out):
        show_doc(str(path), "add")
    text = out.getvalue()
    assert "Location:" in text
    assert "Docs" in text
    assert "Examples/tests" in text
    assert "assert add(1, 2) == 3" in text
    assert "Caller usages:" in text

    out = _StringIO()
    with _redirect_stdout(out):
        returned = nb_overview(str(path))
    text = out.getvalue()
    assert returned == text.strip()
    assert "## Math" in text
    assert "Notebook-level note." not in text
    assert "import math" in text
    assert "def add(a, b):" in text
    assert "Add values." in text
    assert "class Calculator:" in text
    assert "Calculate values." in text
    assert "    def total(self, value):" in text
    assert "Add value to the base." in text
    assert "1 |" not in text

    out = _StringIO()
    with _redirect_stdout(out):
        returned = nb_overview(str(path), verbose=False)
    assert out.getvalue() == ""
    assert "## Math" in returned

    out = _StringIO()
    with _redirect_stdout(out):
        nb_overview(str(path), include_docs=True)
    assert "Notebook-level note." in out.getvalue()

    out = _StringIO()
    with _redirect_stdout(out):
        nb_chapter(str(path), name="Math")
    text = out.getvalue()
    assert "import math" in text
    assert "Tiny rationale." in text
    assert "assert add(1, 2) == 3" in text
    assert "1 |" not in text

    out = _StringIO()
    with _redirect_stdout(out):
        nb_cell(str(path), id=nb.cells[3].id)
    text = out.getvalue()
    assert "## Math" in text
    assert "assert add(1, 2) == 3" in text
    assert "Caller usages:" in text
    assert "1 |" in text
finally:
    remove_demo_path(path)

### Symbol documentation

`show_doc` answers a different question from the focused readers: "what should I know before changing this symbol?" It finds the cell that defines a function, class, or patched method, then collects the surrounding prose and examples.